# Prompt lab — Mellum on llama.cpp

Test prompts against the local model served by **llama-server on `127.0.0.1:8001`** (OpenAI-compatible API).

- Make sure the server is up: `uv run scripts/backend.py start` (from the repo root), health at `http://127.0.0.1:8001/health`.
- The server runs a **single slot** (`-np 1`), so requests serialize — one prompt at a time.
- Mellum is a **thinking** model: its reasoning comes back on `reasoning_content` and the answer on `content`.
- Sampling defaults below match the shipped config: **temperature 1.0 / top_p 0.95 / top_k 20**.

Run the cells top to bottom, then edit `ask(...)` calls to try your own prompts.

In [9]:
from openai import OpenAI

BASE_URL = "http://127.0.0.1:8001/v1"
MODEL = "Mellum2.1-12B-A2.5B-Thinking"

client = OpenAI(base_url=BASE_URL, api_key="not-needed")

# Sanity check: what the server is serving.
[m.id for m in client.models.list().data]

['Mellum2.1-12B-A2.5B-Thinking']

In [10]:
def ask(user, system=None, *, temperature=1.0, top_p=0.95, top_k=20,
        max_tokens=1024, show_thinking=True, **extra):
    """Send one prompt to the local Mellum model and print its reply.

    Sampling defaults match the shipped config (temp 1.0 / top_p 0.95 / top_k 20).
    `top_k` and any other `extra` kwargs go through llama.cpp's extra_body.
    Returns the raw response so you can inspect `.usage`, `.choices`, etc.
    """
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": user})
    resp = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens,
        extra_body={"top_k": top_k, **extra},
    )
    msg = resp.choices[0].message
    think = getattr(msg, "reasoning_content", None)
    if show_thinking and think:
        print("── thinking ──")
        print(think.strip())
        print("── answer ──")
    print((msg.content or "").strip())
    u = resp.usage
    print(f"\n[tokens: prompt {u.prompt_tokens}, completion {u.completion_tokens}]")
    return resp

## Try a prompt

Edit the `system` and `user` strings, re-run, and watch the reasoning and answer.

In [25]:
_ = ask(
    "Write a Python function that returns the nth Fibonacci number. Think briefly, no internal debating, then give the complete answer.",
    system="You are a terse senior Python engineer. Return only final code.",
)

── thinking ──
We need to write a Python function that returns nth Fibonacci number. Should be terse. Probably handle n starting from 0 or 1? Usually Fibonacci sequence: F(0)=0, F(1)=1. We'll implement efficiently maybe iterative. No extra text, just code. The user says "Return only final code." So we output just the code block? Probably just the function definition. Could be:

def fib(n):
    a,b=0,1
    for _ in range(n):
        a,b=b,a+b
    return a

That's it. Ensure it's terse. No imports. So final answer: code block with function.
── answer ──
```python
def fib(n):
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a
```

[tokens: prompt 52, completion 207]


In [ ]:
# Turn sampling down for a more deterministic answer, hide the thinking:
_ = ask(
    "In one sentence, what does a FastAPI RedirectResponse with status 303 do?",
    temperature=0.2, show_thinking=False,
)

## Stream tokens as they arrive

In [ ]:
stream = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Count from 1 to 5, one number per line."}],
    temperature=0.7, max_tokens=128, extra_body={"top_k": 20}, stream=True,
)
for chunk in stream:
    piece = getattr(chunk.choices[0].delta, "content", None)
    if piece:
        print(piece, end="")